# 19 用高级 API 重写分类模型

本节目标：使用 `nn.Flatten`、`nn.Linear`、`nn.CrossEntropyLoss` 和优化器重写 softmax 回归。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset

try:
    from torchvision import datasets, transforms
    HAS_TORCHVISION = True
except Exception as exc:
    HAS_TORCHVISION = False
    TORCHVISION_ERROR = exc


def load_fashion_mnist(train_limit=1024, test_limit=512, batch_size=256):
    data_root = Path("../data")
    if HAS_TORCHVISION:
        try:
            transform = transforms.ToTensor()
            train_full = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
            test_full = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)
            train_data = Subset(train_full, range(min(train_limit, len(train_full))))
            test_data = Subset(test_full, range(min(test_limit, len(test_full))))
            source = "Fashion-MNIST"
        except Exception as exc:
            print("Fashion-MNIST 下载或读取失败，改用同 shape 的随机数据继续练习:", repr(exc))
            train_images = torch.rand(train_limit, 1, 28, 28)
            train_labels = torch.randint(0, 10, (train_limit,))
            test_images = torch.rand(test_limit, 1, 28, 28)
            test_labels = torch.randint(0, 10, (test_limit,))
            train_data = TensorDataset(train_images, train_labels)
            test_data = TensorDataset(test_images, test_labels)
            source = "synthetic fallback"
    else:
        print("torchvision 不可用，改用同 shape 的随机数据继续练习:", repr(TORCHVISION_ERROR))
        train_images = torch.rand(train_limit, 1, 28, 28)
        train_labels = torch.randint(0, 10, (train_limit,))
        test_images = torch.rand(test_limit, 1, 28, 28)
        test_labels = torch.randint(0, 10, (test_limit,))
        train_data = TensorDataset(train_images, train_labels)
        test_data = TensorDataset(test_images, test_labels)
        source = "synthetic fallback"

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, source

import pandas as pd
from torch import nn

torch.manual_seed(42)

train_loader, test_loader, source = load_fashion_mnist(train_limit=1024, test_limit=512, batch_size=256)
print("数据来源:", source)

model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.3)


def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def evaluate(data_loader):
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            batch_size = len(y_batch)
            total_loss += loss.item() * batch_size
            total_acc += accuracy(logits, y_batch) * batch_size
            total_count += batch_size
    model.train()
    return total_loss / total_count, total_acc / total_count

logs = []
for epoch in range(3):
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0
    for X_batch, y_batch in train_loader:
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = len(y_batch)
        total_loss += loss.item() * batch_size
        total_acc += accuracy(logits, y_batch) * batch_size
        total_count += batch_size

    _, test_acc = evaluate(test_loader)
    row = {
        "epoch": epoch + 1,
        "train_loss": total_loss / total_count,
        "train_acc": total_acc / total_count,
        "test_acc": test_acc,
    }
    logs.append(row)
    print(row)

log_table = pd.DataFrame(logs)
print("\n训练日志:")
print(log_table)

## 从零实现 vs API 实现

| 部分 | 从零实现 | API 实现 | API 隐藏的细节 |
|---|---|---|---|
| 展平图像 | `X.reshape((-1, 784))` | `nn.Flatten()` | batch 维保持、特征维展开 |
| 线性分类器 | `X @ W + b` | `nn.Linear(784, 10)` | 参数创建、矩阵乘法、偏置 |
| 交叉熵 | `log_softmax` + 取真实类别 | `nn.CrossEntropyLoss()` | 数值稳定、批量平均 |
| 参数更新 | 手动更新 `W` 和 `b` | `torch.optim.SGD` | 参数遍历、无梯度更新 |
| 训练模式 | 手动控制较少 | `model.train()` / `model.eval()` | 层行为切换 |

API 实现更短、更稳；从零实现更适合理解底层发生了什么。